In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark=SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/04 08:24:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data = [
    (1, "Alice", "Sales", 5000),
    (2, "Bob", "IT", 6000),
    (3, "Charlie", "Finance", 5500),
    (4, "Diana", "Sales", 6500),
    (5, "Evan", "IT", 7000)
]

columns = ["id", "name", "dept", "salary"]
df = spark.createDataFrame(data, columns)
df.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|  Sales|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|  Sales|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [3]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)



In [4]:
df.describe().show()

26/07/04 08:25:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 2:>                                                          (0 + 2) / 2]

+-------+------------------+-----+-------+-----------------+
|summary|                id| name|   dept|           salary|
+-------+------------------+-----+-------+-----------------+
|  count|                 5|    5|      5|                5|
|   mean|               3.0| NULL|   NULL|           6000.0|
| stddev|1.5811388300841898| NULL|   NULL|790.5694150420948|
|    min|                 1|Alice|Finance|             5000|
|    max|                 5| Evan|  Sales|             7000|
+-------+------------------+-----+-------+-----------------+



In [5]:
df.select("name", "salary").show()


+-------+------+
|   name|salary|
+-------+------+
|  Alice|  5000|
|    Bob|  6000|
|Charlie|  5500|
|  Diana|  6500|
|   Evan|  7000|
+-------+------+



In [6]:
df_renamed = df.withColumnRenamed("dept", "department")
df_renamed.show()

+---+-------+----------+------+
| id|   name|department|salary|
+---+-------+----------+------+
|  1|  Alice|     Sales|  5000|
|  2|    Bob|        IT|  6000|
|  3|Charlie|   Finance|  5500|
|  4|  Diana|     Sales|  6500|
|  5|   Evan|        IT|  7000|
+---+-------+----------+------+



In [7]:
df.filter(df.salary > 6000).show()

+---+-----+-----+------+
| id| name| dept|salary|
+---+-----+-----+------+
|  4|Diana|Sales|  6500|
|  5| Evan|   IT|  7000|
+---+-----+-----+------+



In [8]:
df.filter((df.salary > 5500) & (df.dept == "IT")).show()

+---+----+----+------+
| id|name|dept|salary|
+---+----+----+------+
|  2| Bob|  IT|  6000|
|  5|Evan|  IT|  7000|
+---+----+----+------+



In [9]:
df.orderBy("salary").show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|  Sales|  5000|
|  3|Charlie|Finance|  5500|
|  2|    Bob|     IT|  6000|
|  4|  Diana|  Sales|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [10]:
df.orderBy(df.salary.desc()).show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  5|   Evan|     IT|  7000|
|  4|  Diana|  Sales|  6500|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  1|  Alice|  Sales|  5000|
+---+-------+-------+------+



In [11]:
from pyspark.sql.functions import col, when, lit

df_updated = df.withColumn("bonus", df.salary * 0.1)
df_updated.show()

+---+-------+-------+------+-----+
| id|   name|   dept|salary|bonus|
+---+-------+-------+------+-----+
|  1|  Alice|  Sales|  5000|500.0|
|  2|    Bob|     IT|  6000|600.0|
|  3|Charlie|Finance|  5500|550.0|
|  4|  Diana|  Sales|  6500|650.0|
|  5|   Evan|     IT|  7000|700.0|
+---+-------+-------+------+-----+



In [12]:
df_cond = df_updated.withColumn(
    "performance",
    when(col("salary") > 6500, "High")
    .when(col("salary") > 5500, "Medium")
    .otherwise("Low")
)
df_cond.show()

+---+-------+-------+------+-----+-----------+
| id|   name|   dept|salary|bonus|performance|
+---+-------+-------+------+-----+-----------+
|  1|  Alice|  Sales|  5000|500.0|        Low|
|  2|    Bob|     IT|  6000|600.0|     Medium|
|  3|Charlie|Finance|  5500|550.0|        Low|
|  4|  Diana|  Sales|  6500|650.0|     Medium|
|  5|   Evan|     IT|  7000|700.0|       High|
+---+-------+-------+------+-----+-----------+



In [13]:
from pyspark.sql.functions import avg

df.groupBy("dept").agg(avg("salary").alias("avg_salary")).show()

[Stage 19:>                                                         (0 + 2) / 2]

+-------+----------+
|   dept|avg_salary|
+-------+----------+
|  Sales|    5750.0|
|     IT|    6500.0|
|Finance|    5500.0|
+-------+----------+



In [14]:
df.groupBy("dept").count().show()

+-------+-----+
|   dept|count|
+-------+-----+
|  Sales|    2|
|     IT|    2|
|Finance|    1|
+-------+-----+



In [15]:
bonus_data = [
    (1, 500),
    (2, 800),
    (3, 700),
    (4, 900),
    (5, 1000)
]

bonus_df = spark.createDataFrame(bonus_data, ["id", "bonus"])

In [16]:
joined_df = df.join(bonus_df, on="id", how="inner")
joined_df.show()

+---+-------+-------+------+-----+
| id|   name|   dept|salary|bonus|
+---+-------+-------+------+-----+
|  1|  Alice|  Sales|  5000|  500|
|  2|    Bob|     IT|  6000|  800|
|  3|Charlie|Finance|  5500|  700|
|  4|  Diana|  Sales|  6500|  900|
|  5|   Evan|     IT|  7000| 1000|
+---+-------+-------+------+-----+



In [17]:
union_df = df.union(df)
union_df.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|  Sales|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|  Sales|  6500|
|  5|   Evan|     IT|  7000|
|  1|  Alice|  Sales|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|  Sales|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [18]:
df_distinct = df.dropDuplicates(["dept"])
df_distinct.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  3|Charlie|Finance|  5500|
|  2|    Bob|     IT|  6000|
|  1|  Alice|  Sales|  5000|
+---+-------+-------+------+



In [19]:
df_null = df.withColumn("dept", when(col("dept") == "Sales", None).otherwise(col("dept")))
df_null.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|   NULL|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|   NULL|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [20]:
df_clean = df_null.na.drop()
df_clean.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [21]:
df_filled = df_null.na.fill({"dept": "Unknown"})
df_filled.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|Unknown|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|Unknown|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [22]:
df_clean = df_null.na.drop()
df_clean.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [23]:
df_filled = df_null.na.fill({"dept": "Unknown"})
df_filled.show()

+---+-------+-------+------+
| id|   name|   dept|salary|
+---+-------+-------+------+
|  1|  Alice|Unknown|  5000|
|  2|    Bob|     IT|  6000|
|  3|Charlie|Finance|  5500|
|  4|  Diana|Unknown|  6500|
|  5|   Evan|     IT|  7000|
+---+-------+-------+------+



In [24]:
df.count()

5

In [25]:
df.explain(True)

== Parsed Logical Plan ==
LogicalRDD [id#0L, name#1, dept#2, salary#3L], false

== Analyzed Logical Plan ==
id: bigint, name: string, dept: string, salary: bigint
LogicalRDD [id#0L, name#1, dept#2, salary#3L], false

== Optimized Logical Plan ==
LogicalRDD [id#0L, name#1, dept#2, salary#3L], false

== Physical Plan ==
*(1) Scan ExistingRDD[id#0L,name#1,dept#2,salary#3L]



In [26]:
df_clean.write.mode("overwrite").csv("output/cleaned_data")

In [27]:
spark.read.csv("output/cleaned_data").show()

+---+-------+-------+----+
|_c0|    _c1|    _c2| _c3|
+---+-------+-------+----+
|  3|Charlie|Finance|5500|
|  5|   Evan|     IT|7000|
|  2|    Bob|     IT|6000|
+---+-------+-------+----+

